# 01 · Extracción y limpieza — Indicadores Macroeconómicos CMF Chile

**Objetivo:** descargar series históricas (2020–2026) de indicadores macroeconómicos chilenos desde la API oficial de la CMF y exportarlos a `data/clean/` para Power BI.

**Fuente:** [api.cmfchile.cl](https://api.cmfchile.cl/) — acceso gratuito con API Key.

| Indicador | Descripción | Frecuencia |
|-----------|-------------|------------|
| UF | Unidad de Fomento | Diaria |
| Dólar | USD/CLP | Hábil diaria |
| Euro | EUR/CLP | Hábil diaria |
| IPC | Índice de Precios al Consumidor | Mensual |
| UTM | Unidad Tributaria Mensual | Mensual |
| TIP | Tasa de Interés Promedio por segmento | Mensual |
| TMC | Tasa Máxima Convencional por segmento | Mensual |

---

## 0 · Configuración

In [ ]:
import os, json, time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY  = os.getenv('CMF_API_KEY')
BASE_URL = 'https://api.cmfchile.cl/api-sbifv3/recursos_api'

RAW_DIR   = Path('..') / 'data' / 'raw'
CLEAN_DIR = Path('..') / 'data' / 'clean'
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

ANIOS = list(range(2020, 2027))
print('API Key cargada:', bool(API_KEY))
print('Años a extraer:', ANIOS)

## 1 · Funciones auxiliares

In [ ]:
def api_get(endpoint: str, reintentos: int = 3) -> dict:
    """Llama a la API CMF con reintentos ante errores de red."""
    for intento in range(reintentos):
        try:
            r = requests.get(
                f'{BASE_URL}/{endpoint}',
                params={'apikey': API_KEY, 'formato': 'json'},
                timeout=30,
            )
            r.raise_for_status()
            return r.json()
        except (requests.ConnectionError, requests.Timeout):
            if intento == reintentos - 1:
                raise
            time.sleep(2 ** intento)


def a_float(valor: str) -> float | None:
    """
    Convierte strings con formato numérico chileno (puntos de miles,
    coma decimal) a float. Devuelve None si no es convertible.
    Decisión: preferimos None sobre 0 para no distorsionar promedios.
    """
    try:
        return float(str(valor).replace('.', '').replace(',', '.'))
    except (ValueError, AttributeError):
        return None


def a_float_tasa(valor: str) -> float | None:
    """
    Convierte las tasas TIP/TMC, cuyos endpoints entregan el valor con
    PUNTO decimal (ej. '42.82' = 42,82%), a diferencia del resto de las
    series que usan coma. Decisión: una función separada evita tratar el
    punto como separador de miles, lo que inflaría las tasas x100.
    """
    try:
        return float(str(valor).strip())
    except (ValueError, AttributeError):
        return None


def guardar_csv(df: pd.DataFrame, nombre: str) -> None:
    """Guarda CSV con BOM UTF-8 para compatibilidad con Excel y Power BI."""
    ruta = CLEAN_DIR / f'{nombre}.csv'
    df.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f'  → {ruta.name}  ({len(df)} filas)')

## 2 · Series diarias/mensuales (UF, Dólar, Euro, IPC, UTM)

In [ ]:
SERIES = [
    ('uf',    'UFs',     'UF'),
    ('dolar', 'Dolares', 'Dólar (USD/CLP)'),
    ('euro',  'Euros',   'Euro (EUR/CLP)'),
    ('ipc',   'IPCs',    'IPC'),
    ('utm',   'UTMs',    'UTM'),
]

resultados = {}

for recurso, clave, desc in SERIES:
    print(f'Extrayendo {desc}...')
    frames = []
    for anio in ANIOS:
        data = api_get(f'{recurso}/{anio}')
        (RAW_DIR / f'{recurso}_{anio}.json').write_text(
            json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        df = pd.DataFrame(data.get(clave, []))
        df['anio'] = anio
        frames.append(df)
    
    resultado = pd.concat(frames, ignore_index=True)
    resultado['fecha'] = pd.to_datetime(resultado['Fecha'])
    resultado['valor'] = resultado['Valor'].apply(a_float)
    resultado = resultado[['fecha', 'valor']]
    
    guardar_csv(resultado, recurso)
    resultados[recurso] = resultado

print('\nListo.')

## 3 · Verificación de calidad — series simples

In [ ]:
for recurso, _, desc in SERIES:
    df = resultados[recurso]
    nulos = df['valor'].isna().sum()
    print(f'{desc:<20} {len(df):>5} filas | nulos: {nulos} | '
          f'rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')

## 4 · Tasas de interés (TIP y TMC)

Estas series tienen una fila por **categoría de operación** y período, por lo que se consultan mes a mes.

Decisiones de limpieza:
- Se mantienen todas las categorías sin filtrar; el filtro se aplica en Power BI según el análisis.
- **Formato decimal distinto:** estos endpoints entregan las tasas con punto decimal (`'42.82'`), a diferencia del resto de las series que usan formato chileno con coma. Se usa `a_float_tasa` en lugar de `a_float` — de lo contrario las tasas quedarían infladas ×100.

In [ ]:
def extraer_tasas(recurso: str, clave: str) -> pd.DataFrame:
    frames = []
    for anio in ANIOS:
        for mes in range(1, 13):
            try:
                data = api_get(f'{recurso}/{anio}/{mes:02d}')
                time.sleep(0.3)  # evitar saturar la API
            except requests.HTTPError:
                continue
            
            (RAW_DIR / f'{recurso}_{anio}_{mes:02d}.json').write_text(
                json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'
            )
            for reg in data.get(clave, []):
                reg['fecha'] = pd.Timestamp(anio, mes, 1)
            frames.extend(data.get(clave, []))
    
    df = pd.DataFrame(frames)
    if df.empty:
        return df
    df['valor'] = df['Valor'].apply(a_float_tasa)
    return df[['fecha', 'Titulo', 'SubTitulo', 'valor']]


print('Extrayendo TIP (puede tardar ~1 min)...')
df_tip = extraer_tasas('tip', 'TIPs')
guardar_csv(df_tip, 'tip')

print('Extrayendo TMC (puede tardar ~1 min)...')
df_tmc = extraer_tasas('tmc', 'TMCs')
guardar_csv(df_tmc, 'tmc')

## 5 · Vista previa de los datos

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Indicadores Macroeconómicos Chile 2020–2026', fontsize=14)

resultados['uf'].set_index('fecha')['valor'].plot(ax=axes[0,0], title='UF (CLP)', color='steelblue')
resultados['dolar'].set_index('fecha')['valor'].plot(ax=axes[0,1], title='Dólar USD/CLP', color='darkorange')
resultados['ipc'].set_index('fecha')['valor'].plot(ax=axes[1,0], title='IPC Mensual (%)', kind='bar', color='seagreen')
resultados['euro'].set_index('fecha')['valor'].plot(ax=axes[1,1], title='Euro EUR/CLP', color='purple')

for ax in axes.flat:
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/clean/preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Vista previa guardada en data/clean/preview.png')

---
**Próximos pasos:** cargar los CSV de `data/clean/` en Power BI Desktop y construir el dashboard con las 4 páginas definidas en el README.